# Biological BMI Paper — Misclassification

***by Kengo Watanabe***  

In this notebook, obesity misclassification is visualized. Besides, the underlying differences are further analyzed with hierarchical clustering. Both sex models are used here.  

Input:  
* MetBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv  
* ChemBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv  
* ProtBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv  
* CombiBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv  
* Cleaned combined omics: 210104_Biological-BMI-paper_RF-imputation_baseline-combiDF-with-RF-imputation.tsv  
* Cleaned metabolomics: 210104_Biological-BMI-paper_RF-imputation_baseline-metDF-with-RF-imputation.tsv  
* Variables of MetBMI models: 210105_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-BothSex.tsv  
* Cleaned proteomics: 210104_Biological-BMI-paper_RF-imputation_baseline-protDF-with-RF-imputation.tsv  
* Variables of ProtBMI models: 210105_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-BothSex.tsv  

Output:  
* Figure 3a, 3b  
* Figure 3d, 3e  
* Supplementary Figure 5b, 5c  
* Intermediate table for other notebook (misclassification label)  

Original notebook:  
* 210105_Biological-BMI-paper_misclassification.ipynb  

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

from sklearn.preprocessing import StandardScaler
from scipy.cluster import hierarchy
from statsmodels.stats import multitest as multi
import matplotlib.patches as mpatches

!conda list

## 1. Obesity classification

### 1-1. Prepare biological BMI dataframe

In [ ]:
#Import and merge BMI and biological BMI
tempL = ['log_BaseBMI', 'BaseBMI']
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
bmiDF = tempDF[tempL]
bmiDF = pd.merge(bmiDF, tempDF.drop(columns=tempL), left_index=True, right_index=True, how='inner')

fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
bmiDF = pd.merge(bmiDF, tempDF.drop(columns=tempL), left_index=True, right_index=True, how='inner')

fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
bmiDF = pd.merge(bmiDF, tempDF.drop(columns=tempL), left_index=True, right_index=True, how='inner')

fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
bmiDF = pd.merge(bmiDF, tempDF.drop(columns=tempL), left_index=True, right_index=True, how='inner')

display(bmiDF)
display(bmiDF.describe())

tempD = {'BMI':'k', 'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g', 'CombiBMI':'m'}
for bmi in tempD.keys():
    print('Skewness of log_Base'+bmi+':', stats.skew(bmiDF['log_Base'+bmi]))
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
for bmi in tempD.keys():
    sns.distplot(bmiDF['log_Base'+bmi], label=bmi, color=tempD[bmi])
sns.despine()
plt.ylabel('Density')
plt.xlabel('BMI [kg/m'+r'$^2$'+'] (log-scale)')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

for bmi in tempD.keys():
    print('Skewness of Base'+bmi+':', stats.skew(bmiDF['Base'+bmi]))
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
for bmi in tempD.keys():
    sns.distplot(bmiDF['Base'+bmi], label=bmi, color=tempD[bmi])
sns.despine()
plt.ylabel('Density')
plt.xlabel('BMI [kg/m'+r'$^2$'+']')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

### 1-2. Obesity classification

In [ ]:
#Obesity classification
tempD = {'BMI':'k', 'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g', 'CombiBMI':'m'}
for bmi in tempD.keys():
    tempL = []
    for value in bmiDF['Base'+bmi].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    bmiDF['Base'+bmi+'_class'] = tempL
    #Confirmation
    print('Base'+bmi+'_class:')
    tempS1 = bmiDF['Base'+bmi+'_class'].value_counts()
    tempDF = pd.DataFrame({'Count':tempS1, 'Percentage':tempS1/len(bmiDF)*100})
    display(tempDF)
    print('')

#Add the covariates info to make summary dataframe
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-combiDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempL = ['Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Race']
bmiDF = pd.merge(bmiDF, tempDF[tempL], left_index=True, right_index=True, how='inner')
display(bmiDF)

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv'
bmiDF.to_csv(fileDir+fileName, index=True)

### 1-3. Misclassification

In [ ]:
#Calculate misclassification rate based on each biological BMI
tempL = ['MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
misclassDF = pd.DataFrame(index=['Overall_count', 'Overall [%]',
                                 'Underweight_count', 'Underweight [%]',
                                 'Normal_count', 'Normal [%]',
                                 'Overweight_count', 'Overweight [%]',
                                 'Obese_count', 'Obese [%]'])
for bBMI in tempL:
    tempDF = bmiDF.loc[bmiDF['Base'+bBMI+'_class']!='NotCalculated'][['BaseBMI_class', 'Base'+bBMI+'_class']]
    counter0 = 0
    counter1 = 0
    counter2 = 0
    counter3 = 0
    counter4 = 0
    for row_n in tempDF.index.tolist():
        if tempDF.loc[row_n, 'BaseBMI_class']!=tempDF.loc[row_n, 'Base'+bBMI+'_class']:
            counter0 += 1#Overall
            if tempDF.loc[row_n, 'BaseBMI_class']=='Underweight':
                counter1 += 1
            elif tempDF.loc[row_n, 'BaseBMI_class']=='Normal':
                counter2 += 1
            elif tempDF.loc[row_n, 'BaseBMI_class']=='Overweight':
                counter3 += 1
            elif tempDF.loc[row_n, 'BaseBMI_class']=='Obese':
                counter4 += 1
            else:#Just in case
                print('Error?')
    tempS = pd.Series([counter0, counter0/len(tempDF)*100,
                       counter1, counter1/len(tempDF.loc[tempDF['BaseBMI_class']=='Underweight'])*100,
                       counter2, counter2/len(tempDF.loc[tempDF['BaseBMI_class']=='Normal'])*100,
                       counter3, counter3/len(tempDF.loc[tempDF['BaseBMI_class']=='Overweight'])*100,
                       counter4, counter4/len(tempDF.loc[tempDF['BaseBMI_class']=='Obese'])*100],
                      index=['Overall_count', 'Overall [%]',
                             'Underweight_count', 'Underweight [%]',
                             'Normal_count', 'Normal [%]',
                             'Overweight_count', 'Overweight [%]',
                             'Obese_count', 'Obese [%]'],
                      name='vs. '+bBMI)
    misclassDF = pd.concat([misclassDF, tempS], axis=1)
display(misclassDF)

#Plot
tempD = {'vs. Metabolomics':'b', 'vs. Proteomics':'r',
         'vs. Clinical labs':'g', 'vs. Combined omics':'m'}
tempDF = misclassDF.loc[['Overall [%]', 'Normal [%]', 'Overweight [%]', 'Obese [%]']]
tempDF.index = ['Overall', 'Normal', 'Overweight', 'Obese']
tempDF.columns = list(tempD.keys())
tempDF = tempDF.reset_index().melt(var_name='Standard', value_name='Misclassification', id_vars='index')
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(6, 4))
sns.barplot(data=tempDF, x='index', y='Misclassification', hue='Standard', dodge=True, edgecolor='black',
            palette=tempD)
sns.despine()
plt.ylabel('Misclassification rate [%]')
plt.xlabel('BMI-based class')
plt.legend(bbox_to_anchor=(0.5, 1), loc='lower center', borderaxespad=0.5, ncol=2)
plt.axvline(x=(0+1)/2, **{'linestyle':'--', 'color':'k'})
plt.axhspan(ymin=28, ymax=48, facecolor='orange', alpha=0.2)
plt.axhline(y=28, **{'linestyle':'-', 'color':'orange'})
plt.axhline(y=48, **{'linestyle':'-', 'color':'orange'})
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_misclassification_misclassification-rate.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 1-4. Difference

In [ ]:
#Calculate rate of delta
tempL = ['MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
for bbmi in tempL:
    #Calculate difference rate
    bmiDF[bbmi+'–BMI'] = (bmiDF['Base'+bbmi] - bmiDF['BaseBMI']) / bmiDF['BaseBMI'] * 100
    print('Skewness of '+bbmi+' difference:', stats.skew(bmiDF[bbmi+'–BMI']))

tempD = {'MetBMI–BMI':'b', 'ProtBMI–BMI':'r', 'ChemBMI–BMI':'g', 'CombiBMI–BMI':'m'}
tempDF = bmiDF[list(tempD.keys())]
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
for col_n in tempD.keys():
    sns.distplot(tempDF[col_n], label=col_n, color=tempD[col_n])
sns.despine()
plt.ylabel('Density')
plt.xlabel('Rate of '+r'$\Delta$'+'BMI [% BMI]')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
tempL = ['MetBMI–BMI', 'ProtBMI–BMI', 'ChemBMI–BMI', 'CombiBMI–BMI', 'BaseBMI_class']
tempDF = bmiDF[tempL]
tempD = {'Underweight':'blue', 'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
sns.set(style='ticks', font='Arial', context='talk')
p = sns.PairGrid(tempDF, hue='BaseBMI_class', hue_order=list(tempD.keys()), palette=tempD,
                 height=2, aspect=1, layout_pad=-0.5)
p.map_lower(sns.scatterplot, edgecolor='0.3', alpha=0.5, s=25)
p.map_diag(sns.distplot, axlabel=False, kde_kws={'alpha':0.8}, hist_kws={'edgecolor':'white', 'alpha':0.5})
for i, j in zip(*np.triu_indices_from(p.axes, 1)):
    p.axes[i, j].set_visible(False)
for i, j in zip(*np.tril_indices_from(p.axes, 0)):
    p.axes[i, j].set(xlim=(-37.5, 87.5), xticks=np.arange(-25, 75.1, 25),
                     ylim=(-37.5, 87.5), yticks=np.arange(-25, 75.1, 25))
for i, j in zip(*np.tril_indices_from(p.axes, -1)):
    p.axes[i, j].grid(axis='both', linestyle='--', color='gray', alpha=0.3)
    #Draw Pearson's r
    pearson_r = tempDF.drop(columns=['BaseBMI_class']).corr(method='pearson').iloc[i, j]
    p.axes[i, j].annotate(r'$r$'+' = '+str(round(pearson_r, 3)), (87.5, 87.5), fontsize='small',
                          verticalalignment='top', horizontalalignment='right')
pl = plt.legend(bbox_to_anchor=(0.8, 4.1), loc='upper right', title='BMI-based class')
#Add sample size in lagend
for row_i in range(len(pl.get_texts())):
    bmi_class = pl.get_texts()[row_i].get_text()
    count = len(tempDF.loc[tempDF['BaseBMI_class']==bmi_class])
    pl.get_texts()[row_i].set_text(bmi_class+' ('+r'$n$'+' = '+str(count)+')')
##Add xy label annotation
label = 'Difference [% BMI]'
p.fig.text(x=0.5, y=-0.015, s=label, fontsize='large',
           verticalalignment='top', horizontalalignment='center')
p.fig.text(x=-0.025, y=0.5, s=label, fontsize='large',
           verticalalignment='center', horizontalalignment='right', rotation='vertical')
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_misclassification_difference.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

In [ ]:
print('Pearson\'s r:')
tempL = ['MetBMI–BMI', 'ProtBMI–BMI', 'ChemBMI–BMI', 'CombiBMI–BMI']
tempDF = bmiDF[tempL]
display(tempDF.corr(method='pearson'))

## 2. Metabolomics space

### 2-1. Standardization

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-metDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempL = ['log_BaseBMI', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Race']
tempDF = tempDF.drop(columns=tempL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
zscoreDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(zscoreDF.describe(include='all'))

### 2-2. Misclassification label

In [ ]:
#Misclassification label dataframe
labelDF = bmiDF.loc[bmiDF['BaseMetBMI_class']!='NotCalculated'][['BaseBMI_class', 'BaseMetBMI_class']]
print('Sample size of Z-score DF:', len(zscoreDF))
print('Sample size of label DF:', len(labelDF))

#Misclassification
tempL = []
for row_n in labelDF.index.tolist():
    if labelDF.loc[row_n, 'BaseBMI_class'] == labelDF.loc[row_n, 'BaseMetBMI_class']:
        tempL.append('Matched')
    else:
        tempL.append('Misclassified')
labelDF['Misclassification'] = tempL

print('\nSummary')
tempL = ['Underweight', 'Normal', 'Overweight', 'Obese']
for BMI_class in tempL:
    print(' • BMI-based class: '+BMI_class)
    tempDF = labelDF.loc[labelDF['BaseBMI_class']==BMI_class]
    print('    - Sample size:', len(tempDF))
    nMisclassified = len(tempDF.loc[tempDF['Misclassification']=='Misclassified'])
    print('    - Misclassified:', nMisclassified, '(', nMisclassified/len(tempDF)*100, '%)')
    tempS = tempDF['BaseMetBMI_class'].value_counts()
    tempDF = pd.DataFrame({'Count': tempS, 'Percentage':tempS/len(tempDF)*100})
    print('    - Detail of MetBMI-based class:')
    display(tempDF)
    print('')

### 2-3. Color label for variables

In [ ]:
#Prepare column color label based on the contribution in LASSO models
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-BothSex.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='MetBMI model':
        tempL.append('NaN')
    else:
        if tempDF.loc[row_n, 'LASSObcoef'] > 0:
            tempL.append('Positive')
        elif tempDF.loc[row_n, 'LASSObcoef'] < 0:
            tempL.append('Negative')
        else:
            tempL.append('Error')
tempDF['Contribution'] = tempL
tempDF = tempDF.loc[tempDF.index!='MetBMI model']
tempD = {'Positive':'tab:red', 'Negative':'tab:blue'}
colorDF = pd.DataFrame({'Contribution\nin LASSO':tempDF['Contribution'].map(tempD)})
display(colorDF)

### 2-4. Hierarchical clustering: variables with non-zero beta-coefficient in ≥6 models and with ≥14% explained variance for BMI in full model (top 15)

In [ ]:
#Import the cleaned OLS regression result for LASSO variable
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_metBMI-BothSex.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
tempDF = tempDF.loc[tempDF['LASSOnZeros']<=4]
tempDF = tempDF.loc[tempDF['R2']>=14]
#tempDF = tempDF.iloc[0:15]
print('nrows:', len(tempDF))
print('Adjusted p-value < 0.05:', len(tempDF.loc[tempDF['AdjPval']<0.05]))
display(tempDF)

#Extract the variables
zscoreDF_select = zscoreDF.loc[:, zscoreDF.columns.isin(tempDF.index)]
print('Confirm Z-score DF:', zscoreDF_select.shape)

#Update color label
var_colorDF = colorDF.loc[colorDF.index.isin(zscoreDF_select.columns)]

In [ ]:
print('All:')
#Visualization
tempD1 = {'Underweight':'tab:blue', 'Normal':'tab:green', 'Overweight':'tab:orange', 'Obese':'tab:red'}
tempD2 = {'Matched':'white', 'Misclassified':'black'}
tempDF1 = pd.DataFrame({'BMI-based class':labelDF['BaseBMI_class'].map(tempD1),
                        'MetBMI-based class':labelDF['BaseMetBMI_class'].map(tempD1),
                        'Misclassification':labelDF['Misclassification'].map(tempD2)})
t_start = time.time()
sns.set(style='ticks', font='Arial', context='notebook')
cm = sns.clustermap(zscoreDF_select, method='ward', metric='euclidean',
                    row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                    row_colors=tempDF1, col_colors=var_colorDF,
                    cbar_pos=(0.12, 0.01, 0.02, 0.1),
                    figsize=(10, 30), **{'center':0, 'vmin':-2.58, 'vmax':2.58})
cm.cax.set_title(r'$Z$'+'-score')
hm = cm.ax_heatmap.get_position()
rd = cm.ax_row_dendrogram.get_position()
cd = cm.ax_col_dendrogram.get_position()
cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.25])
plt.show()
t_elapsed = time.time() - t_start
print('• Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
#Statistical test
for nclusters in range(2, 6):
    print('• nCluster: '+str(nclusters))
    ##Perform hierarchical clustering for rows: public_client_id
    tempDF1 = pd.DataFrame(hierarchy.linkage(zscoreDF_select, method='ward', metric='euclidean'))
    ##Extract and merge cluster label
    tempS = pd.Series(hierarchy.fcluster(tempDF1, nclusters, criterion='maxclust'),
                      index=zscoreDF_select.index, name='Cluster', dtype=str)
    tempS = 'Cluster_'+tempS
    tempDF = pd.merge(labelDF, tempS, left_index=True, right_index=True)
    ##To check which cluster corresponds to sns.clustermap
    tempL1 = []
    for row_i in range(len(cm.ax_heatmap.get_yticklabels())):
        tempL1.append(cm.ax_heatmap.get_yticklabels()[row_i].get_text())
    tempDF1 = tempDF.loc[tempL1]
    print('  - For checking cluster label')
    for cluster in tempDF['Cluster'].unique().tolist():
        print('     - '+cluster+' includes', tempDF1.loc[tempDF1['Cluster']==cluster].index.tolist()[0:5])
    ##Summary
    tempDF = pd.crosstab(tempDF['Misclassification'], tempDF['Cluster'], margins=True)
    print('  - Misclassification vs. Cluster')
    display(tempDF)
    ##Fisher's exact test for each cluster
    tempDF = tempDF.drop(index=['All'])
    tempS = pd.Series(index=tempDF.drop(columns=['All']).columns, name='Pval')#For p-values
    for cluster in tempS.index.tolist():
        #Prepare 2x2 matrix
        tempDF1 = pd.DataFrame({cluster:tempDF[cluster],
                                'Others':tempDF['All']-tempDF[cluster]})
        if tempDF1.shape == (2, 2):
            #Fisher's exact test
            tempS[cluster] = stats.fisher_exact(tempDF1, alternative='two-sided')[1]
        else:
            tempS[cluster] = np.nan
    ##Multiple tests correction
    tempDF = pd.DataFrame({'Pval':tempS,
                           'AdjPval':multi.multipletests(tempS, alpha=0.05, method='bonferroni',
                                                         is_sorted=False, returnsorted=False)[1]})
    print('  - Fisher\'s exact test (two-sided, Bonferroni correction): cluster vs. the other clusters')
    display(tempDF)
    print('')
print('')

#BMI-based class
tempL = ['Normal', 'Overweight', 'Obese']
for BMI_class in tempL:
    print('BMI-based class: '+BMI_class)
    labelDF_temp = labelDF.loc[labelDF['BaseBMI_class']==BMI_class]
    tempDF = zscoreDF_select.loc[zscoreDF_select.index.isin(labelDF_temp.index)]
    #Visualization
    tempD1 = {'Underweight':'tab:blue', 'Normal':'tab:green', 'Overweight':'tab:orange', 'Obese':'tab:red'}
    tempD2 = {'Matched':'white', 'Misclassified':'black'}
    tempDF1 = pd.DataFrame({'BMI-based class':labelDF_temp['BaseBMI_class'].map(tempD1),
                            'MetBMI-based class':labelDF_temp['BaseMetBMI_class'].map(tempD1),
                            'Misclassification':labelDF_temp['Misclassification'].map(tempD2)})
    t_start = time.time()
    sns.set(style='ticks', font='Arial', context='notebook')
    cm = sns.clustermap(tempDF, method='ward', metric='euclidean',
                        row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                        row_colors=tempDF1, col_colors=var_colorDF,
                        cbar_pos=(0.12, 0.02, 0.02, 0.2),
                        figsize=(10, 15), **{'center':0, 'vmin':-2.58, 'vmax':2.58})
    cm.cax.set_title(r'$Z$'+'-score')
    hm = cm.ax_heatmap.get_position()
    rd = cm.ax_row_dendrogram.get_position()
    cd = cm.ax_col_dendrogram.get_position()
    cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
    cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
    cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.25])
    plt.show()
    t_elapsed = time.time() - t_start
    print('• Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    #Statistical test
    for nclusters in range(2, 6):
        print('• nCluster: '+str(nclusters))
        ##Perform hierarchical clustering for rows: public_client_id
        tempDF1 = pd.DataFrame(hierarchy.linkage(tempDF, method='ward', metric='euclidean'))
        ##Extract and merge cluster label
        tempS = pd.Series(hierarchy.fcluster(tempDF1, nclusters, criterion='maxclust'),
                          index=tempDF.index, name='Cluster', dtype=str)
        tempS = 'Cluster_'+tempS
        tempDF2 = pd.merge(labelDF_temp, tempS, left_index=True, right_index=True)
        ##To check which cluster corresponds to sns.clustermap
        tempL1 = []
        for row_i in range(len(cm.ax_heatmap.get_yticklabels())):
            tempL1.append(cm.ax_heatmap.get_yticklabels()[row_i].get_text())
        tempDF1 = tempDF2.loc[tempL1]
        print('  - For checking cluster label')
        for cluster in tempDF2['Cluster'].unique().tolist():
            print('     - '+cluster+' includes', tempDF1.loc[tempDF1['Cluster']==cluster].index.tolist()[0:5])
        ##Summary
        tempDF1 = pd.crosstab(tempDF2['Misclassification'], tempDF2['Cluster'], margins=True)
        print('  - Misclassification vs. Cluster')
        display(tempDF1)
        ##Fisher's exact test for each cluster
        tempDF1 = tempDF1.drop(index=['All'])
        tempS = pd.Series(index=tempDF1.drop(columns=['All']).columns, name='Pval')#For p-values
        for cluster in tempS.index.tolist():
            #Prepare 2x2 matrix
            tempDF2 = pd.DataFrame({cluster:tempDF1[cluster],
                                    'Others':tempDF1['All']-tempDF1[cluster]})
            if tempDF2.shape == (2, 2):
                #Fisher's exact test
                tempS[cluster] = stats.fisher_exact(tempDF2, alternative='two-sided')[1]
            else:
                tempS[cluster] = np.nan
        ##Multiple tests correction
        tempDF1 = pd.DataFrame({'Pval':tempS,
                               'AdjPval':multi.multipletests(tempS, alpha=0.05, method='bonferroni',
                                                             is_sorted=False, returnsorted=False)[1]})
        print('  - Fisher\'s exact test (two-sided, Bonferroni correction): cluster vs. the other clusters')
        display(tempDF1)
        print('')
    print('')

In [ ]:
#Plot for paper
tempL = ['Normal', 'Obese']
for BMI_class in tempL:
    print('BMI-based class: '+BMI_class)
    labelDF_temp = labelDF.loc[labelDF['BaseBMI_class']==BMI_class]
    tempDF = zscoreDF_select.loc[zscoreDF_select.index.isin(labelDF_temp.index)]
    #Visualization
    tempD1 = {'Underweight':'tab:blue', 'Normal':'tab:green', 'Overweight':'tab:orange', 'Obese':'tab:red'}
    tempD2 = {'Matched':'white', 'Misclassified':'black'}
    tempDF1 = pd.DataFrame({'BMI-based class':labelDF_temp['BaseBMI_class'].map(tempD1),
                            'MetBMI-based class':labelDF_temp['BaseMetBMI_class'].map(tempD1),
                            'Misclassification':labelDF_temp['Misclassification'].map(tempD2)})
    t_start = time.time()
    sns.set(style='ticks', font='Arial', context='talk')
    cm = sns.clustermap(tempDF, method='ward', metric='euclidean',
                        row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                        row_colors=tempDF1, col_colors=var_colorDF, xticklabels=True, yticklabels=False,
                        dendrogram_ratio=(0.2, 0.1), colors_ratio=(0.045, 0.03),
                        cbar_pos=(0.85, 0.43, 0.3, 0.02), cbar_kws={"orientation": "horizontal"},
                        figsize=(8, 13.45), **{'center':0, 'vmin':-2.58, 'vmax':2.58})
    cm.cax.set_title(r'$Z$'+'-score', size='medium', verticalalignment='bottom')
    cm.cax.tick_params(labelsize='small')
    bottom, top = cm.ax_heatmap.get_ylim()
    cm.ax_heatmap.set_ylim(bottom + 0.5, top - 0.5)##To avoid half cut of first and last rows
    hm = cm.ax_heatmap.get_position()
    rd = cm.ax_row_dendrogram.get_position()
    cd = cm.ax_col_dendrogram.get_position()
    cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
    cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0, rd.y0, rd.width, rd.height])
    cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.5])
    #cm.ax_col_colors.yaxis.set_ticks_position('left')
    ##row/column color bar legend (axis is same with cm.cax!)
    row_legend1 = mpatches.Patch(color='tab:green', label='Normal')
    row_legend2 = mpatches.Patch(color='tab:orange', label='Overweight')
    row_legend3 = mpatches.Patch(color='tab:red', label='Obese')
    legend1 = plt.legend(handles=[row_legend1, row_legend2, row_legend3], fontsize='medium',
                         title='BMI/MetBMI-based class', title_fontsize='medium',
                         bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=3.5, frameon=False)
    plt.gca().add_artist(legend1)
    row_legend = mpatches.Patch(color='black', label='Misclassified')
    legend2 = plt.legend(handles=[row_legend], fontsize='medium',
                         title='Misclassification', title_fontsize='medium',
                         bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=9.5, frameon=False)
    plt.gca().add_artist(legend2)
    col_legend1 = mpatches.Patch(color='tab:red', label='Positive')
    col_legend2 = mpatches.Patch(color='tab:blue', label='Negative')
    plt.legend(handles=[col_legend1, col_legend2], fontsize='medium',
               title='Contribution in LASSO', title_fontsize='medium',
               bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=15, frameon=False)
    ##Save
    fileDir = './ExportFigures/'
    fileName = '210105_Biological-BMI-paper_misclassification_hierarchical-clustering_metDF-'+BMI_class+'.tif'
    plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                      pil_kwargs={'compression':'tiff_lzw'})
    plt.show()
    t_elapsed = time.time() - t_start
    print('• Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    print('')

## 3. Proteomics space

### 3-1. Standardization

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-protDF-with-RF-imputation.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempL = ['log_BaseBMI', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Race']
tempDF = tempDF.drop(columns=tempL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
zscoreDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(zscoreDF.describe(include='all'))

### 3-2. Misclassification label

In [ ]:
#Misclassification label dataframe
labelDF = bmiDF.loc[bmiDF['BaseProtBMI_class']!='NotCalculated'][['BaseBMI_class', 'BaseProtBMI_class']]
print('Sample size of Z-score DF:', len(zscoreDF))
print('Sample size of label DF:', len(labelDF))

#Misclassification
tempL = []
for row_n in labelDF.index.tolist():
    if labelDF.loc[row_n, 'BaseBMI_class'] == labelDF.loc[row_n, 'BaseProtBMI_class']:
        tempL.append('Matched')
    else:
        tempL.append('Misclassified')
labelDF['Misclassification'] = tempL

print('\nSummary')
tempL = ['Underweight', 'Normal', 'Overweight', 'Obese']
for BMI_class in tempL:
    print(' • BMI-based class: '+BMI_class)
    tempDF = labelDF.loc[labelDF['BaseBMI_class']==BMI_class]
    print('    - Sample size:', len(tempDF))
    nMisclassified = len(tempDF.loc[tempDF['Misclassification']=='Misclassified'])
    print('    - Misclassified:', nMisclassified, '(', nMisclassified/len(tempDF)*100, '%)')
    tempS = tempDF['BaseProtBMI_class'].value_counts()
    tempDF = pd.DataFrame({'Count': tempS, 'Percentage':tempS/len(tempDF)*100})
    print('    - Detail of ProtBMI-based class:')
    display(tempDF)
    print('')

### 3-3. Color label for variables

In [ ]:
#Prepare column color label based on the contribution in LASSO models
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-BothSex.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
tempL = []
for row_n in tempDF.index.tolist():
    if row_n=='ProtBMI model':
        tempL.append('NaN')
    else:
        if tempDF.loc[row_n, 'LASSObcoef'] > 0:
            tempL.append('Positive')
        elif tempDF.loc[row_n, 'LASSObcoef'] < 0:
            tempL.append('Negative')
        else:
            tempL.append('Error')
tempDF['Contribution'] = tempL
tempDF = tempDF.loc[tempDF.index!='ProtBMI model']
tempD = {'Positive':'tab:red', 'Negative':'tab:blue'}
colorDF = pd.DataFrame({'Contribution\nin LASSO':tempDF['Contribution'].map(tempD)})
display(colorDF)

### 3-4. Hierarchical clustering: variables with non-zero beta-coefficient in ≥8 models and with ≥10% explained variance for BMI in full model (top 15)

In [ ]:
#Import the cleaned OLS regression result for LASSO variable
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_LASSO-bcoef_OLS_protBMI-BothSex.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Variable')
tempDF = tempDF.loc[tempDF['LASSOnZeros']<=2]
tempDF = tempDF.loc[tempDF['R2']>=10]
#tempDF = tempDF.iloc[0:15]
print('nrows:', len(tempDF))
print('Adjusted p-value < 0.05:', len(tempDF.loc[tempDF['AdjPval']<0.05]))
display(tempDF)

#Extract the variables
zscoreDF_select = zscoreDF.loc[:, zscoreDF.columns.isin(tempDF.index)]
print('Confirm Z-score DF:', zscoreDF_select.shape)

#Update color label
var_colorDF = colorDF.loc[colorDF.index.isin(zscoreDF_select.columns)]

In [ ]:
print('All:')
#Visualization
tempD1 = {'Underweight':'tab:blue', 'Normal':'tab:green', 'Overweight':'tab:orange', 'Obese':'tab:red'}
tempD2 = {'Matched':'white', 'Misclassified':'black'}
tempDF1 = pd.DataFrame({'BMI-based class':labelDF['BaseBMI_class'].map(tempD1),
                        'ProtBMI-based class':labelDF['BaseProtBMI_class'].map(tempD1),
                        'Misclassification':labelDF['Misclassification'].map(tempD2)})
t_start = time.time()
sns.set(style='ticks', font='Arial', context='notebook')
cm = sns.clustermap(zscoreDF_select, method='ward', metric='euclidean',
                    row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                    row_colors=tempDF1, col_colors=var_colorDF,
                    cbar_pos=(0.12, 0.825, 0.15, 0.01), cbar_kws={"orientation": "horizontal"},
                    figsize=(10, 24), **{'center':0, 'vmin':-2.58, 'vmax':2.58})
cm.cax.set_title(r'$Z$'+'-score')
hm = cm.ax_heatmap.get_position()
rd = cm.ax_row_dendrogram.get_position()
cd = cm.ax_col_dendrogram.get_position()
cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.25])
plt.show()
t_elapsed = time.time() - t_start
print('• Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
#Statistical test
for nclusters in range(2, 6):
    print('• nCluster: '+str(nclusters))
    ##Perform hierarchical clustering for rows: public_client_id
    tempDF1 = pd.DataFrame(hierarchy.linkage(zscoreDF_select, method='ward', metric='euclidean'))
    ##Extract and merge cluster label
    tempS = pd.Series(hierarchy.fcluster(tempDF1, nclusters, criterion='maxclust'),
                      index=zscoreDF_select.index, name='Cluster', dtype=str)
    tempS = 'Cluster_'+tempS
    tempDF = pd.merge(labelDF, tempS, left_index=True, right_index=True)
    ##To check which cluster corresponds to sns.clustermap
    tempL1 = []
    for row_i in range(len(cm.ax_heatmap.get_yticklabels())):
        tempL1.append(cm.ax_heatmap.get_yticklabels()[row_i].get_text())
    tempDF1 = tempDF.loc[tempL1]
    print('  - For checking cluster label')
    for cluster in tempDF['Cluster'].unique().tolist():
        print('     - '+cluster+' includes', tempDF1.loc[tempDF1['Cluster']==cluster].index.tolist()[0:5])
    ##Summary
    tempDF = pd.crosstab(tempDF['Misclassification'], tempDF['Cluster'], margins=True)
    print('  - Misclassification vs. Cluster')
    display(tempDF)
    ##Fisher's exact test for each cluster
    tempDF = tempDF.drop(index=['All'])
    tempS = pd.Series(index=tempDF.drop(columns=['All']).columns, name='Pval')#For p-values
    for cluster in tempS.index.tolist():
        #Prepare 2x2 matrix
        tempDF1 = pd.DataFrame({cluster:tempDF[cluster],
                                'Others':tempDF['All']-tempDF[cluster]})
        if tempDF1.shape == (2, 2):
            #Fisher's exact test
            tempS[cluster] = stats.fisher_exact(tempDF1, alternative='two-sided')[1]
        else:
            tempS[cluster] = np.nan
    ##Multiple tests correction
    tempDF = pd.DataFrame({'Pval':tempS,
                           'AdjPval':multi.multipletests(tempS, alpha=0.05, method='bonferroni',
                                                         is_sorted=False, returnsorted=False)[1]})
    print('  - Fisher\'s exact test (two-sided, Bonferroni correction): cluster vs. the other clusters')
    display(tempDF)
    print('')
print('')

#BMI-based class
tempL = ['Normal', 'Overweight', 'Obese']
for BMI_class in tempL:
    print('BMI-based class: '+BMI_class)
    labelDF_temp = labelDF.loc[labelDF['BaseBMI_class']==BMI_class]
    tempDF = zscoreDF_select.loc[zscoreDF_select.index.isin(labelDF_temp.index)]
    #Visualization
    tempD1 = {'Underweight':'tab:blue', 'Normal':'tab:green', 'Overweight':'tab:orange', 'Obese':'tab:red'}
    tempD2 = {'Matched':'white', 'Misclassified':'black'}
    tempDF1 = pd.DataFrame({'BMI-based class':labelDF_temp['BaseBMI_class'].map(tempD1),
                            'ProtBMI-based class':labelDF_temp['BaseProtBMI_class'].map(tempD1),
                            'Misclassification':labelDF_temp['Misclassification'].map(tempD2)})
    t_start = time.time()
    sns.set(style='ticks', font='Arial', context='notebook')
    cm = sns.clustermap(tempDF, method='ward', metric='euclidean',
                        row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                        row_colors=tempDF1, col_colors=var_colorDF,
                        cbar_pos=(0.12, 0.825, 0.15, 0.02), cbar_kws={"orientation": "horizontal"},
                        figsize=(10, 12), **{'center':0, 'vmin':-2.58, 'vmax':2.58})
    cm.cax.set_title(r'$Z$'+'-score')
    hm = cm.ax_heatmap.get_position()
    rd = cm.ax_row_dendrogram.get_position()
    cd = cm.ax_col_dendrogram.get_position()
    cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
    cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0.5, rd.y0, rd.width*0.5, rd.height])
    cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.25])
    plt.show()
    t_elapsed = time.time() - t_start
    print('• Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    #Statistical test
    for nclusters in range(2, 6):
        print('• nCluster: '+str(nclusters))
        ##Perform hierarchical clustering for rows: public_client_id
        tempDF1 = pd.DataFrame(hierarchy.linkage(tempDF, method='ward', metric='euclidean'))
        ##Extract and merge cluster label
        tempS = pd.Series(hierarchy.fcluster(tempDF1, nclusters, criterion='maxclust'),
                          index=tempDF.index, name='Cluster', dtype=str)
        tempS = 'Cluster_'+tempS
        tempDF2 = pd.merge(labelDF_temp, tempS, left_index=True, right_index=True)
        ##To check which cluster corresponds to sns.clustermap
        tempL1 = []
        for row_i in range(len(cm.ax_heatmap.get_yticklabels())):
            tempL1.append(cm.ax_heatmap.get_yticklabels()[row_i].get_text())
        tempDF1 = tempDF2.loc[tempL1]
        print('  - For checking cluster label')
        for cluster in tempDF2['Cluster'].unique().tolist():
            print('     - '+cluster+' includes', tempDF1.loc[tempDF1['Cluster']==cluster].index.tolist()[0:5])
        ##Summary
        tempDF1 = pd.crosstab(tempDF2['Misclassification'], tempDF2['Cluster'], margins=True)
        print('  - Misclassification vs. Cluster')
        display(tempDF1)
        ##Fisher's exact test for each cluster
        tempDF1 = tempDF1.drop(index=['All'])
        tempS = pd.Series(index=tempDF1.drop(columns=['All']).columns, name='Pval')#For p-values
        for cluster in tempS.index.tolist():
            #Prepare 2x2 matrix
            tempDF2 = pd.DataFrame({cluster:tempDF1[cluster],
                                    'Others':tempDF1['All']-tempDF1[cluster]})
            if tempDF2.shape == (2, 2):
                #Fisher's exact test
                tempS[cluster] = stats.fisher_exact(tempDF2, alternative='two-sided')[1]
            else:
                tempS[cluster] = np.nan
        ##Multiple tests correction
        tempDF1 = pd.DataFrame({'Pval':tempS,
                               'AdjPval':multi.multipletests(tempS, alpha=0.05, method='bonferroni',
                                                             is_sorted=False, returnsorted=False)[1]})
        print('  - Fisher\'s exact test (two-sided, Bonferroni correction): cluster vs. the other clusters')
        display(tempDF1)
        print('')
    print('')

In [ ]:
#Plot for paper
tempL = ['Normal', 'Obese']
for BMI_class in tempL:
    print('BMI-based class: '+BMI_class)
    labelDF_temp = labelDF.loc[labelDF['BaseBMI_class']==BMI_class]
    tempDF = zscoreDF_select.loc[zscoreDF_select.index.isin(labelDF_temp.index)]
    #Visualization
    tempD1 = {'Underweight':'tab:blue', 'Normal':'tab:green', 'Overweight':'tab:orange', 'Obese':'tab:red'}
    tempD2 = {'Matched':'white', 'Misclassified':'black'}
    tempDF1 = pd.DataFrame({'BMI-based class':labelDF_temp['BaseBMI_class'].map(tempD1),
                            'ProtBMI-based class':labelDF_temp['BaseProtBMI_class'].map(tempD1),
                            'Misclassification':labelDF_temp['Misclassification'].map(tempD2)})
    t_start = time.time()
    sns.set(style='ticks', font='Arial', context='talk')
    cm = sns.clustermap(tempDF, method='ward', metric='euclidean',
                        row_cluster=True, col_cluster=True, row_linkage=None, col_linkage=None,
                        row_colors=tempDF1, col_colors=var_colorDF, xticklabels=True, yticklabels=False,
                        dendrogram_ratio=(0.2, 0.1), colors_ratio=(0.045, 0.03),
                        cbar_pos=(0.85, 0.575, 0.3, 0.02), cbar_kws={"orientation": "horizontal"},
                        figsize=(8, 10), **{'center':0, 'vmin':-2.58, 'vmax':2.58})
    cm.cax.set_title(r'$Z$'+'-score', size='medium', verticalalignment='bottom')
    cm.cax.tick_params(labelsize='small')
    bottom, top = cm.ax_heatmap.get_ylim()
    cm.ax_heatmap.set_ylim(bottom + 0.5, top - 0.5)##To avoid half cut of first and last rows
    hm = cm.ax_heatmap.get_position()
    rd = cm.ax_row_dendrogram.get_position()
    cd = cm.ax_col_dendrogram.get_position()
    cm.ax_heatmap.set_position([hm.x0, hm.y0, hm.width, hm.height])
    cm.ax_row_dendrogram.set_position([rd.x0+rd.width*0, rd.y0, rd.width, rd.height])
    cm.ax_col_dendrogram.set_position([cd.x0, cd.y0, cd.width, cd.height*0.5])
    #cm.ax_col_colors.yaxis.set_ticks_position('left')
    ##row/column color bar legend (axis is same with cm.cax!)
    row_legend1 = mpatches.Patch(color='tab:green', label='Normal')
    row_legend2 = mpatches.Patch(color='tab:orange', label='Overweight')
    row_legend3 = mpatches.Patch(color='tab:red', label='Obese')
    legend1 = plt.legend(handles=[row_legend1, row_legend2, row_legend3], fontsize='medium',
                         title='BMI/ProtBMI-based class', title_fontsize='medium',
                         bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=3.5, frameon=False)
    plt.gca().add_artist(legend1)
    row_legend = mpatches.Patch(color='black', label='Misclassified')
    legend2 = plt.legend(handles=[row_legend], fontsize='medium',
                         title='Misclassification', title_fontsize='medium',
                         bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=9.5, frameon=False)
    plt.gca().add_artist(legend2)
    col_legend1 = mpatches.Patch(color='tab:red', label='Positive')
    col_legend2 = mpatches.Patch(color='tab:blue', label='Negative')
    plt.legend(handles=[col_legend1, col_legend2], fontsize='medium',
               title='Contribution in LASSO', title_fontsize='medium',
               bbox_to_anchor=(0.5, 0), loc='upper center', borderaxespad=15, frameon=False)
    ##Save
    fileDir = './ExportFigures/'
    fileName = '210105_Biological-BMI-paper_misclassification_hierarchical-clustering_protDF-'+BMI_class+'.tif'
    plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                      pil_kwargs={'compression':'tiff_lzw'})
    plt.show()
    t_elapsed = time.time() - t_start
    print('• Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    print('')

## — End of this notebook —